# 🎤 super_Voz - StyleTTS2 Kaggle Pipeline

Este notebook executa o treinamento completo do StyleTTS2 no Kaggle seguindo a mesma lógica do notebook Colab: prepara o ambiente, clona/atualiza o repositório e executa o script real com logs em streaming.

### 🚀 Como usar no Kaggle:
1. Em **Settings**, ative **Internet** e selecione uma **GPU**.
2. Adicione seus áudios como Kaggle Dataset em `/kaggle/input/super-voz/Audios_brutos` ou `/kaggle/input/super-voz/Audios_Brutos`.
3. Opcional: para upload TeraBox, crie Kaggle Secrets `TERABOX_NDUS`, `TERABOX_JS_TOKEN`, `TERABOX_CSRF_TOKEN`, `TERABOX_BROWSER_ID` e `TERABOX_NDUT_FMT`.
4. A seção `terabox` em `styletts2_kaggle_config.yml` usa um wrapper configurável de upload. Sem esses secrets, o pipeline segue com ZIP nos outputs do Kaggle.
5. Durante o treino, checkpoints são salvos em `/kaggle/working/StyleTTS2/Models/super_Voz` e, se TeraBox estiver configurado, enviados periodicamente para o remoto.
6. Ao final, o notebook gera `/kaggle/working/super_voz_resultados.zip` com checkpoints e dataset preparado.
7. Use **Save Version -> Save & Run All (Commit)** para deixar o Kaggle executando mesmo com o navegador fechado.

### Observação sobre Drive/TeraBox
Kaggle não oferece `google.colab.drive.mount()`. O TeraBox também não tem uma CLI oficial estável equivalente ao `rclone`; por isso esta integração usa comandos configuráveis e autenticação por cookie de sessão via Kaggle Secrets. O cookie `ndus` é uma credencial: não coloque esse valor no notebook nem no Git.

### Política Cloudflare
Este notebook preserva `cloudflare_r2` para permitir download dos áudios pelo Cloudflare R2, mas remove `output_prefix` e grava `disable_r2_uploads: true` na configuração runtime para bloquear upload/sync de checkpoints, dataset e resultados para o R2. A saída atual é manter os arquivos em `/kaggle/working`, gerar um ZIP final e tentar upload TeraBox somente quando os secrets de sessão estiverem configurados.

### Se o Kaggle nao importar o notebook
Use o arquivo `run_kaggle_styletts2_celulas.md`: crie um notebook novo no Kaggle, copie as celulas na ordem indicada e depois execute **Run All**. Essa alternativa evita o erro da versao antiga antiga do bootstrap.


In [ ]:
# @title 🚀 INICIAR TUDO (ONE-CLICK)
from pathlib import Path
import os
import shutil
import subprocess
import sys

# No Kaggle, o Resemble Enhance costuma aumentar risco de OOM; deixe "1" se quiser forçar.
os.environ.setdefault("SUPER_VOZ_ENABLE_RESEMBLE", "0")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")


def load_kaggle_secret_to_env(secret_name="TERABOX_NDUS", env_name=None):
    env_name = env_name or secret_name
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(secret_name)
    except Exception as exc:
        print(f"Secret {secret_name} nao encontrado ou indisponivel ({exc}).")
        return False

    if value:
        os.environ[env_name] = value
        print(f"Secret {secret_name} carregado para {env_name}.")
        return True
    return False


def load_runtime_secrets():
    secret_map = {
        "TERABOX_NDUS": "TERABOX_NDUS",
        "TERABOX_JS_TOKEN": "TERABOX_JS_TOKEN",
        "TERABOX_CSRF_TOKEN": "TERABOX_CSRF_TOKEN",
        "TERABOX_BROWSER_ID": "TERABOX_BROWSER_ID",
        "TERABOX_NDUT_FMT": "TERABOX_NDUT_FMT",
        "R2_ENDPOINT_URL": "R2_ENDPOINT_URL",
        "R2_BUCKET_NAME": "R2_BUCKET_NAME",
        "R2_ACCESS_KEY_ID": "R2_ACCESS_KEY_ID",
        "R2_SECRET_ACCESS_KEY": "R2_SECRET_ACCESS_KEY",
        "R2_RAW_AUDIO_PREFIX": "R2_RAW_AUDIO_PREFIX",
    }
    for secret_name, env_name in secret_map.items():
        load_kaggle_secret_to_env(secret_name, env_name)


def setup_environment():
    repo_url = "https://github.com/warllemedicao/Voz_styllett2.git"
    repo_base_dir = Path("/kaggle/working/Super_voz")

    print(f"--- 1. Clonando/Atualizando Repositório: {repo_url} ---")
    if repo_base_dir.exists():
        subprocess.run(["git", "-C", str(repo_base_dir), "fetch", "--all"], check=False)
        subprocess.run(["git", "-C", str(repo_base_dir), "checkout", "main"], check=False)
        subprocess.run(["git", "-C", str(repo_base_dir), "reset", "--hard", "origin/main"], check=False)
    else:
        subprocess.run(["git", "clone", repo_url, str(repo_base_dir)], check=True)

    print("--- 2. Localizando diretório do projeto ---")
    target_script = "run_kaggle_styletts2.py"
    found_path = None
    for path in repo_base_dir.rglob(target_script):
        found_path = path.parent.parent
        break

    project_dir = (found_path or repo_base_dir).resolve()
    os.chdir(project_dir)
    print(f"Projeto: {project_dir}")

    print("--- 3. Instalando dependências críticas ---")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "boto3"], check=True)

    print("--- 4. Verificando Hardware e Aceleração ---")
    try:
        import torch
        if torch.cuda.is_available():
            print(f"GPU Detectada: {torch.cuda.get_device_name(0)}")
        else:
            print("AVISO: GPU não detectada. Ative GPU em Settings antes de rodar o treino.")
    except Exception as exc:
        print(f"Verificação de GPU adiada para o pipeline: {exc}")

    print("--- 5. Política de persistência ---")
    print("Kaggle não monta Google Drive como Colab. Upload/sync para Cloudflare R2 fica desligado nesta execução para evitar taxas.")
    print("Download de áudios do Cloudflare R2 continua permitido se cloudflare_r2 estiver configurado.")
    print("TeraBox será usado somente se os secrets TERABOX_* de sessão estiverem configurados.")

    return project_dir


def make_no_cloud_config(project_dir: Path) -> str:
    import yaml

    source_config = project_dir / "styletts2_kaggle_config.yml"
    runtime_config = project_dir / "styletts2_kaggle_sem_cloudflare.yml"
    with source_config.open("r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f) or {}

    r2_cfg = cfg.get("cloudflare_r2", {}) or {}
    r2_cfg.pop("output_prefix", None)
    r2_cfg["disable_r2_uploads"] = True
    cfg["cloudflare_r2"] = r2_cfg
    cfg["raw_audio_candidates"] = [
        "Audios_brutos",
        "/kaggle/input/super-voz/Audios_brutos",
        "/kaggle/input/super-voz/Audios_Brutos",
    ]
    cfg["styletts2_restore_candidates"] = [
        "/kaggle/input/styllet2",
        "/kaggle/input/styletts2",
        "/kaggle/input/terabox/StyleTTS2",
        "/kaggle/input/super-voz/StyleTTS2",
    ]
    cfg["output_dir"] = "/kaggle/working/super_Voz_outputs"

    tb_cfg = cfg.get("terabox", {}) or {}
    tb_cfg["enabled"] = bool(tb_cfg.get("enabled") or os.environ.get(tb_cfg.get("ndus_env", "TERABOX_NDUS")))
    cfg["terabox"] = tb_cfg
    if tb_cfg["enabled"]:
        print("Config runtime: TeraBox ativado.")
        missing_tb = [name for name in tb_cfg.get("required_env", []) if not os.environ.get(name)]
        if missing_tb:
            print(f"AVISO: TeraBox sem todos os secrets: {', '.join(missing_tb)}")
    else:
        print("Config runtime: TeraBox autenticado desativado; sera tentada restauracao via Kaggle Input styllet2/styletts2.")

    with runtime_config.open("w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)

    print(f"Config runtime com download R2 permitido e upload R2 bloqueado: {runtime_config}")
    return runtime_config.name


def package_outputs() -> Path:
    package_root = Path("/kaggle/working/super_voz_resultados")
    if package_root.exists():
        shutil.rmtree(package_root)
    package_root.mkdir(parents=True, exist_ok=True)

    items = {
        Path("/kaggle/working/StyleTTS2/Models/super_Voz"): package_root / "checkpoints",
        Path("/kaggle/working/super_Voz_styletts2_data"): package_root / "dataset_styletts2",
        Path("/kaggle/working/super_Voz_outputs"): package_root / "outputs",
    }
    for src, dst in items.items():
        if src.exists():
            shutil.copytree(src, dst, dirs_exist_ok=True)

    archive_base = Path("/kaggle/working/super_voz_resultados")
    zip_path = shutil.make_archive(str(archive_base), "zip", root_dir=package_root)
    print(f"Pacote final: {zip_path}")
    print("Se TeraBox nao estiver configurado, baixe esse zip nos outputs do Kaggle e envie pelo app/site da sua conta.")
    return Path(zip_path)


PROJECT_DIR = setup_environment()
os.chdir(PROJECT_DIR)
load_runtime_secrets()

CONFIG_NAME = make_no_cloud_config(PROJECT_DIR)
cmd = [sys.executable, "-u", "scripts/run_kaggle_styletts2.py", "--config", CONFIG_NAME]

print("="*60)
print("INICIANDO PIPELINE SUPER_VOZ")
print("="*60)
print(f"Comando: {' '.join(cmd)}")

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")

code = proc.wait()
if code != 0:
    print(f"\n[ERRO] O pipeline falhou com código {code}.")
else:
    print("\nPipeline finalizado com sucesso!")

package_outputs()
